In [1]:
import pandas as pd
from pathlib import Path
import sys
import numpy as np

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
import mlflow
import mlflow.sklearn
from xgboost import XGBClassifier


In [3]:
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [4]:
from src.pipeline import build_dataset

In [5]:
dataset = build_dataset()

In [6]:
dataset.head()

,Recency_Dias,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,gender,age,city,...,Diversidad_Categorias,Moda_Categoria,Moda_Payment,Moda_Shipping,Ticket_Promedio,Ticket_Desvio,Compras_Ultimos30,Compras_Ultimos90,Promedio_Shipping_Cost,churn
0,189,2,133.85,1,1,1,3,F,23,Paysandú,...,2,Snacks,credit_card,standard,66.925000,16.638223,0,0,6.075000,1
1,480,4,235.11,1,1,1,3,F,49,Montevideo,...,2,Beverages,debit_card,standard,58.777500,36.255436,0,0,7.157500,1
2,6,72,3626.60,4,5,5,14,F,44,Rivera,...,5,Household,credit_card,standard,50.369444,28.968426,3,9,5.625694,0
3,2,63,3335.50,5,5,5,15,F,31,Las Piedras,...,5,Beverages,credit_card,standard,52.944444,28.036658,2,7,5.375714,0
4,1,75,4414.55,5,5,5,15,M,26,Salto,...,5,Technology,credit_card,standard,58.860667,22.508065,3,7,5.151333,0


## Primer modelo. Random forest

In [6]:
mlflow.set_experiment("churn")

2025/11/11 12:36:09 INFO mlflow.tracking.fluent: Experiment with name 'churn' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///c:/Users/sebastian.celesia/Desktop/GestionDatos/notebooks/mlruns/795828195094325042', creation_time=1762875369150, experiment_id='795828195094325042', last_update_time=1762875369150, lifecycle_stage='active', name='churn', tags={}>

In [18]:
dataset.columns

Index(['Recency_Dias', 'Frequency', 'Monetary', 'R_Score', 'F_Score',
       'M_Score', 'RFM_Score', 'gender', 'age', 'city', 'country',
       'citizenship', 'registration_date', 'last_seen',
       'Promedio_Dias_Compras', 'Varianza_Dias_Compras',
       'Max_Dias_Entre_Compras', 'Min_Dias_Entre_Compras', 'Tenure_Dias',
       'Diversidad_Categorias', 'Moda_Categoria', 'Moda_Payment',
       'Moda_Shipping', 'Ticket_Promedio', 'Ticket_Desvio',
       'Compras_Ultimos30', 'Compras_Ultimos90', 'Promedio_Shipping_Cost',
       'churn'],
      dtype='object')

In [7]:
x = dataset.drop(columns=["churn","registration_date","last_seen"])
y = dataset["churn"]

In [8]:
numeric_features = [
    "Recency_Dias", "Frequency", "Monetary",
    "R_Score", "F_Score", "M_Score", "RFM_Score",
    "age",
    "Promedio_Dias_Compras", "Varianza_Dias_Compras",
    "Max_Dias_Entre_Compras", "Min_Dias_Entre_Compras",
    "Tenure_Dias", "Diversidad_Categorias",
    "Ticket_Promedio", "Ticket_Desvio",
    "Compras_Ultimos30", "Compras_Ultimos90",
    "Promedio_Shipping_Cost",]

categorical_features = [
    "gender", "city", "country", "citizenship",
    "Moda_Categoria", "Moda_Payment", "Moda_Shipping",]

X_train, X_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42,
    stratify=y)

In [9]:
preprocess_tree = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ],
    remainder="passthrough"  
)

In [11]:
with mlflow.start_run(run_name="rf_baseline"):
    # 1) Definir modelo y pipeline (TU código)
    rf_model = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    )

    rf_pipeline = Pipeline(steps=[
        ("preprocess", preprocess_tree),  
        ("model", rf_model),
    ])

    # 2) Entrenar
    rf_pipeline.fit(X_train, y_train)

    # 3) Predicciones
    y_pred_rf  = rf_pipeline.predict(X_test)
    y_proba_rf = rf_pipeline.predict_proba(X_test)[:, 1]

    # 4) Métricas
    accuracy  = accuracy_score(y_test, y_pred_rf)
    precision = precision_score(y_test, y_pred_rf)  
    recall    = recall_score(y_test, y_pred_rf)
    f1        = f1_score(y_test, y_pred_rf)
    roc_auc   = roc_auc_score(y_test, y_proba_rf)

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1       : {f1:.4f}")
    print(f"ROC AUC  : {roc_auc:.4f}  ({roc_auc*100:.2f}%)")

    # 5) Loguear PARÁMETROS del modelo
    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("n_estimators", rf_model.n_estimators)
    mlflow.log_param("max_depth", rf_model.max_depth)
    mlflow.log_param("class_weight", rf_model.class_weight)
    mlflow.log_param("n_features", X_train.shape[1])

    # 6) Loguear MÉTRICAS
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1", f1)
    mlflow.log_metric("roc_auc", roc_auc)

    # 7) Guardar el PIPELINE completo (preprocess + modelo)
    mlflow.sklearn.log_model(rf_pipeline, artifact_path="model")

Accuracy : 0.8422
Precision: 0.7364
Recall   : 0.6585
F1       : 0.6953
ROC AUC  : 0.8831  (88.31%)


c:\Users\sebastian.celesia\Desktop\GestionDatos\.venv\Lib\site-packages\_distutils_hack\__init__.py:18: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
c:\Users\sebastian.celesia\Desktop\GestionDatos\.venv\Lib\site-packages\_distutils_hack\__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")
2025/11/11 12:42:07 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided when logging the model.


Para el primer modelo se entrenó un Random Forest classifier utilizando como variables explicativas los indicadores RFM (Recency_Dias, Frequency, Monetary, RFM_Score) junto con variables demográficas (gender, age, city, country, citizenship) y de comportamiento transaccional (Promedio_Dias_Compras, Diversidad_Categorias, Moda_Payment, Ticket_Promedio, Compras_Ultimos30/90, etc.), teniendo como variable objetivo la columna churn.
Dado que se trata de un problema de churn con clases desbalanceadas (30% clientes que abandonan y 70% que permanecen), más que en la exactitud global nos centramos en las métricas asociadas a la clase positiva. En particular, priorizamos el recall de los churners, ya que para el negocio es más costoso no identificar a un cliente en riesgo (falso negativo) que actuar sobre un cliente que finalmente no se va (falso positivo).
Complementariamente, utilizamos el F1-score para medir el equilibrio entre precisión y recall y el ROC AUC para evaluar la capacidad discriminativa general del modelo.

## modelos XGboost